# 02 - Database queries

The cleaned NSL-KDD records live in a normalized SQLite schema (see `sql/schema.sql`):

* `protocols`, `services`, `flags`, `attack_types`: dimension tables
* `connections`: fact table, one row per connection
* `v_connections_full`: view that joins them all

This notebook runs the analytical queries that we cite in the report straight against that schema.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
from sqlalchemy import text
from ids_pipeline import config
from ids_pipeline.schema import get_engine

engine = get_engine(config.DB_URL)

def q(sql, **params):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

## Q1. Connection counts per attack family

In [ ]:
q("""
SELECT split, attack_family, COUNT(*) AS n
FROM v_connections_full
GROUP BY split, attack_family
ORDER BY split, n DESC;
""")

## Q2. Top-10 services targeted by attacks

In [ ]:
q("""
SELECT service, COUNT(*) AS n_attacks
FROM v_connections_full
WHERE split = 'train' AND is_attack = 1
GROUP BY service ORDER BY n_attacks DESC LIMIT 10;
""")

## Q3. Average bytes per protocol and family

In [ ]:
q("""
SELECT protocol_type, attack_family,
       COUNT(*) AS n,
       ROUND(AVG(src_bytes),2) AS avg_src_bytes,
       ROUND(AVG(dst_bytes),2) AS avg_dst_bytes
FROM v_connections_full
WHERE split = 'train'
GROUP BY protocol_type, attack_family
ORDER BY protocol_type, attack_family;
""")

## Q4. Failed-login profile per family (R2L signal)

In [ ]:
q("""
SELECT attack_family,
       ROUND(AVG(num_failed_logins),3) AS avg_failed_logins,
       ROUND(AVG(logged_in),3) AS pct_logged_in,
       ROUND(AVG(is_guest_login),3) AS pct_guest
FROM v_connections_full WHERE split='train' GROUP BY attack_family;
""")

## Q5. Connection-rate profile (Probe / DoS signal)

In [ ]:
q("""
SELECT attack_family,
       ROUND(AVG(count),2) AS avg_count,
       ROUND(AVG(srv_count),2) AS avg_srv_count,
       ROUND(AVG(serror_rate),3) AS avg_serror_rate,
       ROUND(AVG(rerror_rate),3) AS avg_rerror_rate
FROM v_connections_full WHERE split='train' GROUP BY attack_family;
""")